In [ ]:
from huggingface_hub import HfApi, create_repo, login
import joblib, json

# --- Mode ---
SAVE_RESULTS = False   # True = merge train+dev, predict test, save to GDrive
                       # False = train on train only, eval on dev (generalization check)

# ABLATION_MODE = True   # True = loop over all aug views, print comparison table at end
MERGE_DEV_TRAIN = True
USE_ENSEMBLE = True
# --- Ensemble models ---|
ENSEMBLE_MODELS = [
    "FacebookAI/xlm-roberta-base",
    "microsoft/mdeberta-v3-base",
    "jhu-clsp/mmBERT-base",
]
RUN_NUM = 3

# --- Aug view ablation ---
# "en_context_only"  : en_context (fallback en_target_word if NaN)
# "en_ctx_plus_tgt"  : en_context [SEP] en_target_word
# "en_target_only"   : en_target_word always
AUG_VIEWS = ["en_target_only"]
# AUG_VIEWS = ["en_context_only", "en_ctx_plus_tgt", "en_target_only", "full_input"]


# --- Languages ---
LANGUAGES = ["es", "de", "cn"]

# --- Submission ---
GDRIVE_SAVE_DIR = "/content/drive/MyDrive/RESEARCH/BEA/submission/xxx"
GDRIVE_FOLDER_ID = "17EYEOeXb3eAFiphJzHEbo4sNU7AYGvQo"

# --- Architecture ---
PROJ_DIM = 128

# --- Contrastive loss ---
LAMBDA_IMP = 0
LAMBDA_SOFT = 0
IMP_TEMPERATURE = 0.01
SOFT_TEMPERATURE = 0.02
SIGMA = 0.5

# --- Training ---
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
SEED = 17

# save pth
HF_REPO_ID = "wicaksonolxn/bea2026-kvl-toebm"  # your mono repo
SAVE_TO_HF = True
HF_TOKEN = os.environ.get('HF_TOKEN', '')  # set via env var or Colab secrets
login(token=HF_TOKEN, add_to_git_credential=False)

In [13]:
def save_model_to_hf(model, tokenizer, repo_id, subfolder, commit_msg=None):
    """Save encoder + heads + tokenizer to HF under repo_id/subfolder."""
    local_dir = f"/tmp/hf_upload/{subfolder}"
    os.makedirs(local_dir, exist_ok=True)

    torch.save(model.state_dict(), os.path.join(local_dir, "pytorch_model.bin"))
    tokenizer.save_pretrained(local_dir)
    model.encoder.config.save_pretrained(local_dir)

    api = HfApi()
    create_repo(repo_id, exist_ok=True, repo_type="model")
    api.upload_folder(
        repo_id=repo_id,
        folder_path=local_dir,
        path_in_repo=subfolder,
        commit_message=commit_msg or f"Upload {subfolder}",
    )
    print(f"  -> Pushed to {repo_id}/{subfolder}")


def save_ridge_to_hf(ridge, repo_id, lang, aug_view):
    """Save Ridge regressor + metadata to HF."""
    subfolder = f"ridge_{lang}_{aug_view}"
    local_dir = f"/tmp/hf_upload/{subfolder}"
    os.makedirs(local_dir, exist_ok=True)

    joblib.dump(ridge, os.path.join(local_dir, "ridge.joblib"))
    meta = {
        "alpha": float(ridge.alpha_),
        "coef": [float(w) for w in ridge.coef_],
        "intercept": float(ridge.intercept_),
        "lang": lang,
        "aug_view": aug_view,
        "ensemble_models": ENSEMBLE_MODELS,
    }
    with open(os.path.join(local_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    api = HfApi()
    api.upload_folder(
        repo_id=repo_id,
        folder_path=local_dir,
        path_in_repo=subfolder,
        commit_message=f"Ridge {lang} {aug_view}",
    )
    print(f"  -> Pushed ridge to {repo_id}/{subfolder}")


In [14]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
import os
import random
import inspect
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    TrainingArguments, Trainer, set_seed
)
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.linear_model import RidgeCV
from scipy.stats import spearmanr, pearsonr

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed(SEED)

from google.colab import drive
drive.mount('/content/drive')

gdrive_csv = lambda u: pd.read_csv(
    f"https://drive.google.com/uc?export=download&id={u.split('/d/')[1].split('/')[0]}"
)

GDRIVE_URLS = {
    "es_train": "https://drive.google.com/file/d/1HHkAKzlSf6VTV8kYlfvHUhFVHEkbiP2p/view?usp=sharing",
    "de_train": "https://drive.google.com/file/d/1EHiiUUPiNHt3QbwaTXHPFZZjfJ3DAa59/view?usp=sharing",
    "cn_train": "https://drive.google.com/file/d/1Uml7gxGaeS5Bn48YGjM8SpSHcf_8bynQ/view?usp=sharing",
    "es_dev": "https://drive.google.com/file/d/1Ec0pvclSrVS9zajxAudf2c5lbxP_HmJL/view?usp=sharing",
    "de_dev": "https://drive.google.com/file/d/1dUgbQkRSmfVALgZyrBIFNmjgRi7s7ibM/view?usp=sharing",
    "cn_dev": "https://drive.google.com/file/d/1r96NcTrhAY7LiGWe5AuEFbp0pb6LbPFY/view?usp=sharing",
    "cn_train_translated": "https://drive.google.com/file/d/1aUHk9mVpDRUt40Otcj1aPSTvZLwiAJwT/view?usp=drive_link",
    "de_train_translated": "https://drive.google.com/file/d/1GKT7d_JPLoFQAxcz26eNtygLzZKNIpg7/view?usp=sharing",
    "es_train_translated": "https://drive.google.com/file/d/1iB-J-6Nz5jZG2Ly9RRDVKI8_lg5k_RaH/view?usp=sharing",
    # Add test URLs when available:
    "es_test": "https://drive.google.com/file/d/1rifhjczUUF6Lt2EwKvFijcV0TCTjTKFO/view?usp=sharing",
    "de_test": "https://drive.google.com/file/d/1NzuaOBWCXbhoxcu_RGXCGlDIM73JTWJQ/view?usp=sharing",
    "cn_test": "https://drive.google.com/file/d/1igvsY6RHDgtVEc6aqeHX672seHS217Rn/view?usp=sharing",
}


def save_predictions(df_preds, lang, run_num, model_tag="single"):
    save_dir = os.path.join(GDRIVE_SAVE_DIR, lang)
    os.makedirs(save_dir, exist_ok=True)
    out_path = os.path.join(save_dir, f"prediction_run_{run_num}.csv")
    df_preds.index.name = "item_id"
    df_preds.columns = ["prediction"]
    df_preds.to_csv(out_path, index=True)
    print(f"Saved: {out_path}")
import torch.multiprocessing as mp

def _worker(mname, lang, df_tr, df_ev, run_tag, queue):
    torch.cuda.set_per_process_memory_fraction(0.30)
    preds, metrics = train_and_predict(mname, lang, df_tr, df_ev, run_tag=run_tag)
    metrics["run_tag"] = run_tag
    queue.put((mname, preds, metrics))

mp.set_start_method("spawn", force=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
class MultiTaskDifficultyModel(nn.Module):

    def __init__(self, pretrained_model, proj_dim, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(pretrained_model)
        hidden_size = self.encoder.config.hidden_size

        # ModernBERT (mmBERT) does not accept token_type_ids
        fwd_params = inspect.signature(self.encoder.forward).parameters
        self.pass_token_type_ids = "token_type_ids" in fwd_params

        self.dropout = nn.Dropout(dropout)
        self.regression_head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 1),
        )
        self.proj_head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, proj_dim),
        )

    def _encode(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if self.pass_token_type_ids and token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**kwargs)
        return self.dropout(outputs.last_hidden_state[:, 0])

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, labels=None):
        cls_hidden = self._encode(input_ids, attention_mask, token_type_ids)
        reg_logits = self.regression_head(cls_hidden).squeeze(-1)
        proj = self.proj_head(cls_hidden)
        return {"reg_logits": reg_logits, "proj": proj, "cls_hidden": cls_hidden}

    def encode(self, input_ids, attention_mask, token_type_ids=None):
        return self._encode(input_ids, attention_mask, token_type_ids)

In [16]:
class ImpConLoss(nn.Module):

    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, h_orig, h_aug):
        B = h_orig.size(0)
        if B <= 1:
            return torch.tensor(0.0, device=h_orig.device)

        features = F.normalize(torch.cat([h_orig, h_aug], dim=0), dim=1)
        sim = torch.matmul(features, features.T) / self.temperature

        positive_mask = torch.zeros(2 * B, 2 * B, device=h_orig.device)
        for i in range(B):
            positive_mask[i, B + i] = 1.0
            positive_mask[B + i, i] = 1.0

        self_mask = torch.eye(2 * B, device=h_orig.device)
        logits_max, _ = sim.max(dim=1, keepdim=True)
        logits = sim - logits_max.detach()
        exp_logits = torch.exp(logits) * (1.0 - self_mask)
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-8)
        mean_log_prob = (positive_mask * log_prob).sum(dim=1)
        return -mean_log_prob.mean()


class SoftSupConLoss(nn.Module):

    def __init__(self, temperature=0.07, sigma=0.5):
        super().__init__()
        self.temperature = temperature
        self.sigma = sigma

    def forward(self, features, scores):
        B = features.size(0)
        if B <= 1:
            return torch.tensor(0.0, device=features.device)

        features = F.normalize(features, dim=1)
        sim = torch.matmul(features, features.T) / self.temperature
        score_diff = (scores.unsqueeze(0) - scores.unsqueeze(1)).abs()
        pos_weights = torch.exp(-score_diff.pow(2) / (2 * self.sigma ** 2))

        self_mask = torch.eye(B, dtype=torch.bool, device=features.device)
        pos_weights = pos_weights.masked_fill(self_mask, 0.0)

        logits_max, _ = sim.max(dim=1, keepdim=True)
        logits = sim - logits_max.detach()
        exp_logits = torch.exp(logits) * (~self_mask).float()
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-8)
        weighted_log_prob = (pos_weights * log_prob).sum(dim=1) / pos_weights.sum(dim=1).clamp(min=1e-8)
        return -weighted_log_prob.mean()

In [17]:
class MultiTaskRegressionTrainer(Trainer):

    def __init__(self, lambda_imp, lambda_soft, imp_temperature=0.07,
                 soft_temperature=0.07, sigma=0.5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_imp = lambda_imp
        self.lambda_soft = lambda_soft
        self.impcon = ImpConLoss(temperature=imp_temperature)
        self.soft_supcon = SoftSupConLoss(temperature=soft_temperature, sigma=sigma)
        self.reg_loss = nn.MSELoss()

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        full_ids = inputs.pop("full_ids")
        full_mask = inputs.pop("full_mask")

        outputs = model(input_ids=full_ids, attention_mask=full_mask)
        preds = outputs["reg_logits"]
        h_orig = outputs["cls_hidden"]
        proj_features = outputs["proj"]

        h_aug = model.encode(inputs["aug_ids"], inputs["aug_mask"])

        l_reg = self.reg_loss(preds, labels)
        l_imp = self.impcon(h_orig, h_aug)
        l_soft = self.soft_supcon(proj_features, labels)
        total_loss = l_reg + self.lambda_imp * l_imp + self.lambda_soft * l_soft

        if return_outputs:
            return total_loss, {"logits": preds}
        return total_loss

In [18]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    labels = labels.squeeze()

    mse = mean_squared_error(labels, predictions)
    rmse = root_mean_squared_error(labels, predictions)
    rho, _ = spearmanr(labels, predictions)
    if np.isnan(rho):
        rho = 0.0

    return {"mse": mse, "rmse": rmse, "spearman": rho}


class MultiTaskCollator:
    def __init__(self, pad_token_id):
        self.pad_id = pad_token_id or 0

    def _pad(self, batch_ids, batch_masks):
        max_len = max(len(ids) for ids in batch_ids)
        p_ids, p_masks = [], []
        for ids, mask in zip(batch_ids, batch_masks):
            pad_len = max_len - len(ids)
            p_ids.append(ids + [self.pad_id] * pad_len)
            p_masks.append(mask + [0] * pad_len)
        return torch.tensor(p_ids), torch.tensor(p_masks)

    def __call__(self, features):
        full_ids, full_mask = self._pad(
            [b["full_input_ids"] for b in features],
            [b["full_attention_mask"] for b in features],
        )
        aug_ids, aug_mask = self._pad(
            [b["aug_input_ids"] for b in features],
            [b["aug_attention_mask"] for b in features],
        )
        return {
            "labels": torch.tensor([f["labels"] for f in features], dtype=torch.float),
            "full_ids": full_ids, "full_mask": full_mask,
            "aug_ids": aug_ids, "aug_mask": aug_mask,
        }


def make_preprocess_fn(tok, aug_view="en_context_only"):
    sep = tok.sep_token or " [SEP] "

    def preprocess(examples):
        texts, texts_aug = [], []
        en_contexts = examples.get("en_context", [None] * len(examples["L1_context"]))

        for w, ctx, clue, tgt, en_ctx in zip(
            examples["L1_source_word"], examples["L1_context"],
            examples["en_target_clue"], examples["en_target_word"], en_contexts
        ):
            full_text = sep.join([w, ctx, clue, tgt])
            texts.append(full_text)

            has_ctx = en_ctx is not None and en_ctx == en_ctx  # NaN check

            if aug_view == "full_input":
                # Aug = same as regression head input (SimCSE-style via dropout)
                texts_aug.append(full_text)
            elif aug_view == "en_target_only":
                texts_aug.append(tgt)
            elif aug_view == "en_ctx_plus_tgt":
                ctx_part = en_ctx if has_ctx else tgt
                texts_aug.append(ctx_part + sep + tgt)
            else:  # en_context_only
                texts_aug.append(en_ctx if has_ctx else tgt)

        full_enc = tok(texts, truncation=True, max_length=128)

        # full_input needs 128; en_ctx_plus_tgt needs 64; rest 32
        aug_max_len = {
            "full_input": 128,
            "en_ctx_plus_tgt": 64,
        }.get(aug_view, 32)
        aug_enc = tok(texts_aug, truncation=True, max_length=aug_max_len)

        return {
            "full_input_ids": full_enc["input_ids"],
            "full_attention_mask": full_enc["attention_mask"],
            "aug_input_ids": aug_enc["input_ids"],
            "aug_attention_mask": aug_enc["attention_mask"],
            "labels": examples["label"],
        }
    return preprocess


def load_lang_data(lang, merge_dev_into_train=False):
    df_train = gdrive_csv(GDRIVE_URLS[f"{lang}_train"])
    df_dev = gdrive_csv(GDRIVE_URLS[f"{lang}_dev"])

    if f"{lang}_train_translated" in GDRIVE_URLS:
        df_trans = gdrive_csv(GDRIVE_URLS[f"{lang}_train_translated"])
        df_train["en_context"] = df_trans["en_context"]

    df_train.rename(columns={"GLMM_score": "label"}, inplace=True)
    df_dev.rename(columns={"GLMM_score": "label"}, inplace=True)
    df_train.set_index("item_id", inplace=True)
    df_dev.set_index("item_id", inplace=True)

    if merge_dev_into_train:
        df_train = pd.concat([df_train, df_dev])
        if "en_context" in df_train.columns:
            df_train["en_context"] = df_train["en_context"].fillna(df_train["en_target_word"])

    test_key = f"{lang}_test"
    if test_key in GDRIVE_URLS:
        df_test = gdrive_csv(GDRIVE_URLS[test_key])
        if "GLMM_score" in df_test.columns:
            df_test.rename(columns={"GLMM_score": "label"}, inplace=True)
        else:
            df_test["label"] = 0.0
        df_test.set_index("item_id", inplace=True)
    else:
        df_test = df_dev

    return df_train, df_dev, df_test

In [19]:
def train_and_predict(model_name, lang, df_train, df_eval, run_tag="model",
                      extra_predict_df=None, aug_view="en_context_only",
                      df_test_for_eval=None):

    tok = AutoTokenizer.from_pretrained(model_name)
    model = MultiTaskDifficultyModel(model_name, proj_dim=PROJ_DIM)
    model.float()

    preprocess_fn = make_preprocess_fn(tok, aug_view=aug_view)
    collator = MultiTaskCollator(tok.pad_token_id)

    ds_train = Dataset.from_pandas(df_train)
    enc_train = ds_train.map(preprocess_fn, batched=True, remove_columns=ds_train.column_names)

    # use test set for checkpoint selection if provided
    eval_df = df_test_for_eval if df_test_for_eval is not None else df_eval
    ds_eval = Dataset.from_pandas(eval_df)
    enc_eval = ds_eval.map(preprocess_fn, batched=True, remove_columns=ds_eval.column_names)

    output_dir = f"./output_{run_tag}_{lang}"
    use_eval = True

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",      # still evaluate on df_test every epoch
        save_strategy="epoch",      # save checkpoints
        logging_strategy="epoch",
        save_total_limit=1,         # keep only the latest checkpoint
        load_best_model_at_end=False,   # important
        metric_for_best_model=None,     # optional, makes intent clear
        greater_is_better=None,         # optional
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        max_grad_norm=MAX_GRAD_NORM,
        report_to="none",
        remove_unused_columns=False,
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
        fp16=False,
        bf16=False,
    )
    trainer = MultiTaskRegressionTrainer(
        model=model,
        args=training_args,
        train_dataset=enc_train,
        eval_dataset=enc_eval,
        processing_class=tok,
        compute_metrics=compute_metrics,
        lambda_imp=LAMBDA_IMP,
        lambda_soft=LAMBDA_SOFT,
        imp_temperature=IMP_TEMPERATURE,
        soft_temperature=SOFT_TEMPERATURE,
        sigma=SIGMA,
        data_collator=collator,
    )

    trainer.train()

    # trainer.model is now the checkpoint with lowest eval_mse
    raw_preds = trainer.predict(enc_eval)
    predictions = raw_preds.predictions.squeeze()

    eval_results = trainer.evaluate(enc_eval)
    metrics = {
        "model": model_name.split("/")[-1],
        "lang": lang,
        "spearman": round(eval_results["eval_spearman"], 4),
        "rmse": round(eval_results["eval_rmse"], 4),
        "mse": round(eval_results["eval_mse"], 4),
    }

    extra_predictions = None
    if extra_predict_df is not None:
        ds_extra = Dataset.from_pandas(extra_predict_df)
        enc_extra = ds_extra.map(preprocess_fn, batched=True, remove_columns=ds_extra.column_names)
        raw_extra = trainer.predict(enc_extra)
        extra_predictions = raw_extra.predictions.squeeze()

    trained_model = trainer.model
    del trainer
    torch.cuda.empty_cache()
    gc.collect()

    return predictions, metrics, extra_predictions, trained_model, tok

In [20]:
if USE_ENSEMBLE:
    ablation_results = []

    for aug_view in AUG_VIEWS:
        all_results = []
        all_ensemble_results = []

        for lang in LANGUAGES:
            print(f"\n{'='*60}")
            print(f"  ENSEMBLE TRAINING | lang={lang} | merge={MERGE_DEV_TRAIN} | aug={aug_view}")
            print(f"{'='*60}\n")

            df_train, df_dev, df_test = load_lang_data(lang, merge_dev_into_train=MERGE_DEV_TRAIN)

            train_preds_list = []
            test_preds_list = []
            dev_preds_list = []

            for mname in ENSEMBLE_MODELS:
                short = mname.split("/")[-1]
                print(f"\n--- Training {short} for {lang} ---")

                if MERGE_DEV_TRAIN:
                    # test_preds via eval dataset, train_preds via extra for Ridge
                    test_preds, metrics, train_preds, trained_model, tok = train_and_predict(
                        mname, lang, df_train, df_test,
                        run_tag=short, extra_predict_df=df_train,
                        aug_view=aug_view,
                        df_test_for_eval=df_test,
                    )
                    train_preds_list.append(train_preds)
                    test_preds_list.append(test_preds)

                    # Upload model to HF
                    if SAVE_TO_HF:
                        subfolder = f"model_{short}_{lang}_{aug_view}"
                        save_model_to_hf(trained_model, tok, HF_REPO_ID, subfolder,
                                         commit_msg=f"{short} {lang} {aug_view} best-on-test")

                    del trained_model, tok
                    torch.cuda.empty_cache()
                    gc.collect()

                    t_labels = df_test["label"].values
                    print(f"  Test {short:<20s} (best ckpt) | spearman={metrics['spearman']:.4f} | rmse={metrics['rmse']:.4f}")

                else:
                    preds, metrics, _, trained_model, tok = train_and_predict(
                        mname, lang, df_train, df_dev, run_tag=short,
                        aug_view=aug_view,
                    )
                    dev_preds_list.append(preds)

                    if SAVE_TO_HF:
                        subfolder = f"model_{short}_{lang}_{aug_view}"
                        save_model_to_hf(trained_model, tok, HF_REPO_ID, subfolder,
                                         commit_msg=f"{short} {lang} {aug_view} best-on-dev")

                    del trained_model, tok
                    torch.cuda.empty_cache()
                    gc.collect()

                all_results.append(metrics)

            # --- Ridge stacking ---
            if MERGE_DEV_TRAIN:
                meta_matrix = np.column_stack(train_preds_list)
                meta_labels = df_train["label"].values

                ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0])
                ridge.fit(meta_matrix, meta_labels)

                print(f"\nRidge fitted | alpha={ridge.alpha_:.4f} | coef={[round(w,4) for w in ridge.coef_]} | intercept={ridge.intercept_:.4f}")

                # Upload Ridge to HF
                if SAVE_TO_HF:
                    save_ridge_to_hf(ridge, HF_REPO_ID, lang, aug_view)

                test_matrix = np.column_stack(test_preds_list)
                blended = ridge.predict(test_matrix)

                if SAVE_RESULTS:
                    df_out = pd.DataFrame({"prediction": blended}, index=df_test.index)
                    save_predictions(df_out, lang, RUN_NUM)

                if "label" in df_test.columns and df_test["label"].abs().sum() > 0:
                    test_labels = df_test["label"].values
                    avg_test = test_matrix.mean(axis=1)
                    print(f"  Test Ridge  | spearman={spearmanr(test_labels, blended)[0]:.4f} | rmse={root_mean_squared_error(test_labels, blended):.4f}")
                    print(f"  Test Avg    | spearman={spearmanr(test_labels, avg_test)[0]:.4f} | rmse={root_mean_squared_error(test_labels, avg_test):.4f}")

                    ablation_results.append({
                        "aug_view": aug_view, "lang": lang, "method": "ridge_stack",
                        "spearman": round(spearmanr(test_labels, blended)[0], 4),
                        "rmse": round(root_mean_squared_error(test_labels, blended), 4),
                    })

            else:
                dev_matrix = np.column_stack(dev_preds_list)
                dev_labels = df_dev["label"].values

                ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0])
                ridge.fit(dev_matrix, dev_labels)

                print(f"\nRidge fitted | alpha={ridge.alpha_:.4f} | coef={[round(w,4) for w in ridge.coef_]} | intercept={ridge.intercept_:.4f}")

                if SAVE_TO_HF:
                    save_ridge_to_hf(ridge, HF_REPO_ID, lang, aug_view)

                blended   = ridge.predict(dev_matrix)
                avg_preds = dev_matrix.mean(axis=1)

                ridge_metrics = {
                    "lang": lang, "method": "ridge_stack",
                    "alpha": round(ridge.alpha_, 4),
                    "weights": [round(w, 4) for w in ridge.coef_],
                    "intercept": round(ridge.intercept_, 4),
                    "spearman": round(spearmanr(dev_labels, blended)[0], 4),
                    "rmse": round(root_mean_squared_error(dev_labels, blended), 4),
                    "mse": round(mean_squared_error(dev_labels, blended), 4),
                }
                avg_metrics = {
                    "lang": lang, "method": "simple_avg",
                    "spearman": round(spearmanr(dev_labels, avg_preds)[0], 4),
                    "rmse": round(root_mean_squared_error(dev_labels, avg_preds), 4),
                    "mse": round(mean_squared_error(dev_labels, avg_preds), 4),
                }
                all_ensemble_results.append(ridge_metrics)
                all_ensemble_results.append(avg_metrics)

                ablation_results.append({**ridge_metrics, "aug_view": aug_view})

        print("\n\n================ INDIVIDUAL MODEL RESULTS ================")
        for r in all_results:
            print(r)

        if all_ensemble_results:
            print("\n================ ENSEMBLE RESULTS ================")
            for r in all_ensemble_results:
                print(r)
        print("=========================================================")

    if ablation_results:
        print(f"\n\n{'='*70}")
        print("  ABLATION SUMMARY")
        print(f"{'='*70}")
        df_abl = pd.DataFrame(ablation_results)
        ridge_df = df_abl[df_abl["method"] == "ridge_stack"][["aug_view", "lang", "spearman", "rmse"]]
        print(ridge_df.to_string(index=False))
        pivot = ridge_df.pivot(index="aug_view", columns="lang", values="spearman")
        pivot["mean"] = pivot.mean(axis=1)
        print("\n--- Spearman pivot ---")
        print(pivot.to_string())


  ENSEMBLE TRAINING | lang=es | merge=True | aug=en_target_only


--- Training xlm-roberta-base for es ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

  -> Pushed to wicaksonolxn/bea2026-kvl-toebm/model_mmBERT-base_cn_en_target_only
  Test mmBERT-base          (best ckpt) | spearman=0.8428 | rmse=0.9107

Ridge fitted | alpha=0.1000 | coef=[np.float64(-0.0079), np.float64(0.0581), np.float64(0.8966)] | intercept=-0.0267


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._target_only/ridge.joblib: 100%|##########|   690B /   690B            

  -> Pushed ridge to wicaksonolxn/bea2026-kvl-toebm/ridge_cn_en_target_only
  Test Ridge  | spearman=0.8484 | rmse=0.8932
  Test Avg    | spearman=0.8561 | rmse=0.9437


================ INDIVIDUAL MODEL RESULTS ================
{'model': 'xlm-roberta-base', 'lang': 'es', 'spearman': 0.767, 'rmse': 1.2799, 'mse': 1.6381}
{'model': 'mdeberta-v3-base', 'lang': 'es', 'spearman': 0.8356, 'rmse': 1.1513, 'mse': 1.3255}
{'model': 'mmBERT-base', 'lang': 'es', 'spearman': 0.8245, 'rmse': 1.079, 'mse': 1.1643}
{'model': 'xlm-roberta-base', 'lang': 'de', 'spearman': 0.7815, 'rmse': 1.2914, 'mse': 1.6676}
{'model': 'mdeberta-v3-base', 'lang': 'de', 'spearman': 0.8333, 'rmse': 1.1593, 'mse': 1.3439}
{'model': 'mmBERT-base', 'lang': 'de', 'spearman': 0.8264, 'rmse': 1.0104, 'mse': 1.0208}
{'model': 'xlm-roberta-base', 'lang': 'cn', 'spearman': 0.7738, 'rmse': 1.2097, 'mse': 1.4633}
{'model': 'mdeberta-v3-base', 'lang': 'cn', 'spearman': 0.8552, 'rmse': 1.0267, 'mse': 1.0542}
{'model': 'mmBERT-base'